## Amazon Product

In [1]:
"""
First we have to make the spark session for our Project 

"""

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
spark = SparkSession.builder \
.appName("Amazon Product Bestseller Churn") \
.getOrCreate()

spark.sparkContext.setLogLevel("WARN")

In [2]:
"""Adding Different important Libraries require for the project"""
from pyspark.sql.functions import col, count, when
from pyspark.sql.functions import col, countDistinct
from pyspark.sql.functions import col, round, when
from pyspark.sql.functions import avg
import matplotlib.pyplot as plt


### Loading the Different DataSets

In [3]:
product_path = r"amazon_products.csv"
category_path = r"amazon_categories.csv"

Product_DF = spark.read.csv(
    product_path,
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"',
    quote='"'
)

Category_DF = spark.read.csv(
    category_path,
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"',
    quote='"'
)


print("Product sell Schema")
Product_DF.printSchema()

print("Category Schema")
Category_DF.printSchema()

Product sell Schema
root
 |-- asin: string (nullable = true)
 |-- title: string (nullable = true)
 |-- imgUrl: string (nullable = true)
 |-- productURL: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- reviews: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- listPrice: double (nullable = true)
 |-- category_id: integer (nullable = true)
 |-- isBestSeller: boolean (nullable = true)
 |-- boughtInLastMonth: integer (nullable = true)

Category Schema
root
 |-- id: integer (nullable = true)
 |-- category_name: string (nullable = true)



In [4]:
print("Product sell Schema")
Product_DF.show(5)

print("Category Schema")
Category_DF.show(5)

Product sell Schema
+----------+--------------------+--------------------+--------------------+-----+-------+------+---------+-----------+------------+-----------------+
|      asin|               title|              imgUrl|          productURL|stars|reviews| price|listPrice|category_id|isBestSeller|boughtInLastMonth|
+----------+--------------------+--------------------+--------------------+-----+-------+------+---------+-----------+------------+-----------------+
|B014TMV5YE|Sion Softside Exp...|https://m.media-a...|https://www.amazo...|  4.5|      0|139.99|      0.0|        104|       false|             2000|
|B07GDLCQXV|Luggage Sets Expa...|https://m.media-a...|https://www.amazo...|  4.5|      0|169.99|   209.99|        104|       false|             1000|
|B07XSCCZYG|Platinum Elite So...|https://m.media-a...|https://www.amazo...|  4.6|      0|365.49|   429.99|        104|       false|              300|
|B08MVFKGJM|Freeform Hardside...|https://m.media-a...|https://www.amazo...|  4.6

In [5]:
"""Checking the Null values in the data set of wach column"""


# Checking for nulls in Product_DF
Product_null_counts = Product_DF.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in Product_DF.columns
])


# Checking for nulls in Category_DF
Category_null_counts = Category_DF.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in Category_DF.columns
])

print("Null count in each column of Product")
Product_null_counts.show()
print("-----------------------------------------------------------------------------------------------------------")
print("Null count in each column of Category")
Category_null_counts.show()

Null count in each column of Product
+----+-----+------+----------+-----+-------+-----+---------+-----------+------------+-----------------+
|asin|title|imgUrl|productURL|stars|reviews|price|listPrice|category_id|isBestSeller|boughtInLastMonth|
+----+-----+------+----------+-----+-------+-----+---------+-----------+------------+-----------------+
|   0|    0|     0|         0|    0|      0|    0|        0|          0|           0|                0|
+----+-----+------+----------+-----+-------+-----+---------+-----------+------------+-----------------+

-----------------------------------------------------------------------------------------------------------
Null count in each column of Category
+---+-------------+
| id|category_name|
+---+-------------+
|  0|            0|
+---+-------------+



### We verified that our dataset contains no null values, we will now examine the unique values in each column.

In [6]:
# Counting unique values for each column in Product_DF
Product_unique_counts = Product_DF.agg(
    *(countDistinct(col(c)).alias(c) for c in Product_DF.columns)
)
print("Unique values count in each column")
Product_unique_counts.show()

Unique values count in each column
+-------+-------+-------+----------+-----+-------+-----+---------+-----------+------------+-----------------+
|   asin|  title| imgUrl|productURL|stars|reviews|price|listPrice|category_id|isBestSeller|boughtInLastMonth|
+-------+-------+-------+----------+-----+-------+-----+---------+-----------+------------+-----------------+
|1426337|1385431|1372162|   1426337|   41|  11861|29961|    14518|        248|           2|               30|
+-------+-------+-------+----------+-----+-------+-----+---------+-----------+------------+-----------------+



In [7]:
# See unique values in the Best Seller column
print("Unique values in isBestSeller:")
Product_DF.select("isBestSeller").distinct().show()

print("Sample of Unique Category IDs:")
Product_DF.select("category_id").distinct().limit(10).show()

print("Sample of Unique stars:")
Product_DF.select("stars").distinct().limit(10).show()

Unique values in isBestSeller:
+------------+
|isBestSeller|
+------------+
|        true|
|       false|
+------------+

Sample of Unique Category IDs:
+-----------+
|category_id|
+-----------+
|        148|
|        243|
|         31|
|        251|
|        137|
|         65|
|         53|
|        255|
|        133|
|         78|
+-----------+

Sample of Unique stars:
+-----+
|stars|
+-----+
|  2.4|
|  0.0|
|  3.5|
|  2.9|
|  3.7|
|  1.4|
|  2.3|
|  4.9|
|  3.1|
|  4.2|
+-----+



In [8]:
# Check unique star ratings available (e.g., are they only 0.5 increments?)
print("Unique Star Ratings:")
Product_DF.select("stars").distinct().orderBy("stars").show()

# Check the spread of unique prices
Product_DF.select("price").summary("min", "25%", "50%", "75%", "max").show()

Unique Star Ratings:
+-----+
|stars|
+-----+
|  0.0|
|  1.0|
|  1.2|
|  1.3|
|  1.4|
|  1.5|
|  1.6|
|  1.7|
|  1.8|
|  1.9|
|  2.0|
|  2.1|
|  2.2|
|  2.3|
|  2.4|
|  2.5|
|  2.6|
|  2.7|
|  2.8|
|  2.9|
+-----+
only showing top 20 rows
+-------+--------+
|summary|   price|
+-------+--------+
|    min|     0.0|
|    25%|   11.99|
|    50%|   19.95|
|    75%|   35.99|
|    max|19731.81|
+-------+--------+



## Now we are going to check the Duplicate values into the dataset

In [9]:
# Count total rows vs unique rows
total_rows = Product_DF.count()
unique_rows = Product_DF.distinct().count()

print(f"Total Rows: {total_rows}")
print(f"Unique Rows: {unique_rows}")
print(f"Duplicate Rows: {total_rows - unique_rows}")

Total Rows: 1426337
Unique Rows: 1426337
Duplicate Rows: 0


### As we see we do not have any duplicate Row in our dataSet and now we will Join the data set 

In [10]:
# 1. Performing the join
# We match 'category_id' from products to 'id' from categories
Joined_DF = Product_DF.join(
    Category_DF, 
    Product_DF.category_id == Category_DF.id, 
    "inner"
)

# 2. Clean up redundant columns
# After joining, you'll have both 'category_id' and 'id' (which are the same). 
# We drop 'id' to keep the DataFrame clean.
Final_DF = Joined_DF.drop("id")

# 3. Verify the result
print(f"Total rows after join: {Final_DF.count()}")
Final_DF.select("asin", "title", "category_name").show(5)

if Final_DF.count() > Product_DF.count():
    print("⚠️ Warning: Row count increased. Check Category_DF for duplicate IDs.")
else:
    print("✅ Join successful: Data integrity maintained.")

Total rows after join: 1426337
+----------+--------------------+-------------+
|      asin|               title|category_name|
+----------+--------------------+-------------+
|B014TMV5YE|Sion Softside Exp...|    Suitcases|
|B07GDLCQXV|Luggage Sets Expa...|    Suitcases|
|B07XSCCZYG|Platinum Elite So...|    Suitcases|
|B08MVFKGJM|Freeform Hardside...|    Suitcases|
|B01DJLKZBA|Winfield 2 Hardsi...|    Suitcases|
+----------+--------------------+-------------+
only showing top 5 rows
✅ Join successful: Data integrity maintained.


In [11]:
def checkShape(df, df_name):
    # Get row count and column count
    row_count = df.count()
    col_count = len(df.columns)
    print(f"DataFrame '{df_name}' has {row_count} rows and {col_count} columns.")

# Now run your code again
checkShape(Final_DF, "master_df")

print("\nFinal Dataset Schema:")
Final_DF.printSchema()

DataFrame 'master_df' has 1426337 rows and 12 columns.

Final Dataset Schema:
root
 |-- asin: string (nullable = true)
 |-- title: string (nullable = true)
 |-- imgUrl: string (nullable = true)
 |-- productURL: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- reviews: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- listPrice: double (nullable = true)
 |-- category_id: integer (nullable = true)
 |-- isBestSeller: boolean (nullable = true)
 |-- boughtInLastMonth: integer (nullable = true)
 |-- category_name: string (nullable = true)



In [12]:
"""
Making additional Feature for better performance 
Also we created some bolean column for best performance 
"""

# Safer calculation using a conditional check
Final_DF = Final_DF.withColumn(
    "savings_pct", 
    when(
        (col("listPrice").isNotNull()) & (col("listPrice") > 0), 
        round(((col("listPrice") - col("price")) / col("listPrice")) * 100, 2)
    ).otherwise(0) # Set to 0 if listPrice is 0 or Null
)

# 1. High Discount Flag (1 if >= 30%, else 0)
Final_DF = Final_DF.withColumn(
    "is_high_discount", 
    when(col("savings_pct") >= 30, 1).otherwise(0)
)

# 2. Highly Rated Flag (1 if stars >= 4.5, else 0)
Final_DF = Final_DF.withColumn(
    "is_highly_rated", 
    when(col("stars") >= 4.5, 1).otherwise(0)
)

Final_DF = Final_DF.withColumn(
    "isBestSeller", 
    col("isBestSeller").cast("int")
)

# Preview the results
Final_DF.select("title", "price", "savings_pct", "is_high_discount", "is_highly_rated","isBestSeller" ).show(20)


+--------------------+------+-----------+----------------+---------------+------------+
|               title| price|savings_pct|is_high_discount|is_highly_rated|isBestSeller|
+--------------------+------+-----------+----------------+---------------+------------+
|Sion Softside Exp...|139.99|        0.0|               0|              1|           0|
|Luggage Sets Expa...|169.99|      19.05|               0|              1|           0|
|Platinum Elite So...|365.49|       15.0|               0|              1|           0|
|Freeform Hardside...|291.59|      17.72|               0|              1|           0|
|Winfield 2 Hardsi...|174.99|      43.55|               1|              1|           0|
|Maxlite 5 Softsid...|144.49|        0.0|               0|              1|           0|
|Hard Shell Carry ...|169.99|        0.0|               0|              1|           0|
|Maxporter II 30" ...|299.99|        0.0|               0|              1|           0|
|Omni 2 Hardside E...|112.63|   

In [13]:
# KPI: What percentage of our catalog is 'Highly Rated'?
rating_kpi = Final_DF.select(avg("is_highly_rated")).collect()[0][0] * 100

print(f"Highly Rated Product KPI: {rating_kpi:.2f}%")

Highly Rated Product KPI: 49.90%


In [14]:
# Removing multiple columns that won't be used in the AI model
cols_to_drop = ["imgUrl", "productURL"]
Final_DF = Final_DF.drop(*cols_to_drop)

print("Final Schema is here")
Final_DF.printSchema()

Final Schema is here
root
 |-- asin: string (nullable = true)
 |-- title: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- reviews: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- listPrice: double (nullable = true)
 |-- category_id: integer (nullable = true)
 |-- isBestSeller: integer (nullable = true)
 |-- boughtInLastMonth: integer (nullable = true)
 |-- category_name: string (nullable = true)
 |-- savings_pct: double (nullable = true)
 |-- is_high_discount: integer (nullable = false)
 |-- is_highly_rated: integer (nullable = false)



In [15]:
print("Final Schema is here")
Final_DF.show(5)



Final Schema is here
+----------+--------------------+-----+-------+------+---------+-----------+------------+-----------------+-------------+-----------+----------------+---------------+
|      asin|               title|stars|reviews| price|listPrice|category_id|isBestSeller|boughtInLastMonth|category_name|savings_pct|is_high_discount|is_highly_rated|
+----------+--------------------+-----+-------+------+---------+-----------+------------+-----------------+-------------+-----------+----------------+---------------+
|B014TMV5YE|Sion Softside Exp...|  4.5|      0|139.99|      0.0|        104|           0|             2000|    Suitcases|        0.0|               0|              1|
|B07GDLCQXV|Luggage Sets Expa...|  4.5|      0|169.99|   209.99|        104|           0|             1000|    Suitcases|      19.05|               0|              1|
|B07XSCCZYG|Platinum Elite So...|  4.6|      0|365.49|   429.99|        104|           0|              300|    Suitcases|       15.0|           

## Now our data is almost ready for the EDA and Preprocessing 
### Now i will save this clean dataset into CSV then we will do our other tasks

i tried so many time to save my joined file into one CSV but i can able to save that into a single sile 
it is because i have 14 million Rows in my dataset and every time i want to save it my Spark goes crashed

### So thats why im now going to make EDA and Preprocessing here so i will blanced my data and all of that or as my dataset is clean and only issue is to blance that so im going to work in this CSV

In [16]:
checkShape(Final_DF, "master_df")
Final_DF.printSchema()

DataFrame 'master_df' has 1426337 rows and 13 columns.
root
 |-- asin: string (nullable = true)
 |-- title: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- reviews: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- listPrice: double (nullable = true)
 |-- category_id: integer (nullable = true)
 |-- isBestSeller: integer (nullable = true)
 |-- boughtInLastMonth: integer (nullable = true)
 |-- category_name: string (nullable = true)
 |-- savings_pct: double (nullable = true)
 |-- is_high_discount: integer (nullable = false)
 |-- is_highly_rated: integer (nullable = false)



In [17]:
print(f"Rows: {Final_DF.count()}, Columns: {len(Final_DF.columns)}")
Final_DF.printSchema()


Rows: 1426337, Columns: 13
root
 |-- asin: string (nullable = true)
 |-- title: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- reviews: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- listPrice: double (nullable = true)
 |-- category_id: integer (nullable = true)
 |-- isBestSeller: integer (nullable = true)
 |-- boughtInLastMonth: integer (nullable = true)
 |-- category_name: string (nullable = true)
 |-- savings_pct: double (nullable = true)
 |-- is_high_discount: integer (nullable = false)
 |-- is_highly_rated: integer (nullable = false)



In [18]:
Final_DF.select(
    "stars",
    "reviews",
    "price",
    "listPrice",
    "savings_pct",
    "boughtInLastMonth"
).describe().show()


+-------+------------------+-----------------+------------------+------------------+------------------+------------------+
|summary|             stars|          reviews|             price|         listPrice|       savings_pct| boughtInLastMonth|
+-------+------------------+-----------------+------------------+------------------+------------------+------------------+
|  count|           1426337|          1426337|           1426337|           1426337|           1426337|           1426337|
|   mean|3.9995118264479737|180.7508197571822| 43.37540368136131|12.449159714715984|5.0566198521049195|141.98229450683814|
| stddev|1.3442923176553476|1761.452959001129|130.28929583115132|46.111984184644875|11.427434190707844| 836.2719652279358|
|    min|               0.0|                0|               0.0|               0.0|           -781.56|                 0|
|    max|               5.0|           346563|          19731.81|            999.99|             100.0|            100000|
+-------+-------

In [19]:
from pyspark.sql.functions import round

Final_DF.groupBy(round("stars", 1).alias("rating")) \
    .count() \
    .orderBy("rating") \
    .show()


+------+------+
|rating| count|
+------+------+
|   0.0|131023|
|   1.0|  4430|
|   1.2|     6|
|   1.3|    36|
|   1.4|    62|
|   1.5|   230|
|   1.6|    60|
|   1.7|    88|
|   1.8|   113|
|   1.9|   146|
|   2.0|  2072|
|   2.1|   280|
|   2.2|   327|
|   2.3|   405|
|   2.4|   542|
|   2.5|  1237|
|   2.6|   861|
|   2.7|  1269|
|   2.8|  1399|
|   2.9|  1940|
+------+------+
only showing top 20 rows


In [20]:
Final_DF.groupBy("isBestSeller") \
    .count() \
    .withColumnRenamed("count", "product_count") \
    .show()


+------------+-------------+
|isBestSeller|product_count|
+------------+-------------+
|           1|         8520|
|           0|      1417817|
+------------+-------------+



### Here we see data we have only 10% of data of the record where the product is ment to be bestseller so we should blanced the data like i will make the data like 2x of the bestseller count in the Non Bestseller recored so we can't lead to the overfitting of our ML models in further.

In [21]:
# Count Best Seller records
best_seller_count = Final_DF.filter(col("isBestSeller") == 1).count()

# Target Non–Best Seller count (2x Best Sellers)
target_non_bs_count = best_seller_count * 2

# Split dataset by class
df_bs = Final_DF.filter(col("isBestSeller") == 1)
df_non_bs = Final_DF.filter(col("isBestSeller") == 0)

# Downsample Non–Best Sellers
df_non_bs_sampled = df_non_bs.sample(
    withReplacement=False,
    fraction=target_non_bs_count / df_non_bs.count(),
    seed=42
)

# Combine balanced dataset
balanced_df = df_bs.union(df_non_bs_sampled)

# Verify class distribution
balanced_df.groupBy("isBestSeller").count().show()


+------------+-----+
|isBestSeller|count|
+------------+-----+
|           1| 8520|
|           0|17222|
+------------+-----+



In [22]:
print(f"Rows: {balanced_df.count()}, Columns: {len(balanced_df.columns)}")

Rows: 25742, Columns: 13


In [23]:
import os

# =========================
# Export the Final Dataset
# =========================

# Step 1: Define full CSV path 
output_csv = os.path.join(os.getcwd(), "balanced_df.csv")

# Step 2: Save full DataFrame using Pandas
balanced_df.toPandas().to_csv(output_csv, index=False)

print(f"DataFrame successfully saved as '{output_csv}'")


DataFrame successfully saved as 'h:\Semester 1\Big Data & AI\Big Data & AI\Amazon Dataset\balanced_df.csv'


# Finally i am able to save my blanced CSV
## NOTE :
### I Did this because i dont have the best machine and im sure in the organization we must have that good machine so we will work on almost all the dataset So for now this is the reason i did this  